# 13E — Independent open-water and flooded-vegetation reference validation

**Priority 5.** Construct a stratified probability sample based on radar prediction strata, then obtain **independent**, date-matched manual labels while blinded to radar class. Accuracy is computed only after reference imagery, source and analyst are documented. Flooded vegetation is a separate class.

**Execution:** This is a source workflow, NOT results. A successful extract needs your own data / Earth Engine session. Do not treat low-VV detections as validated water, flood depth or methane flux.

In [1]:
# Every notebook uses the same ROOT; no hard-coded drive or EE project ID.
from pathlib import Path
import os, sys
starting = Path(os.environ.get("HAKALUKI_ROOT", Path.cwd())).expanduser().resolve()
for candidate in [starting, *starting.parents]:
    if (candidate/"src/analysis13/core.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Extract the ZIP into the project root and open Jupyter there, or set HAKALUKI_ROOT")
sys.path.insert(0, str(ROOT))
print("Research root:", ROOT)


Research root: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project


## Select the actual SAR raster and corresponding source scene

Run 13B first. Choose a cached matched four-band raster and its exact scene ID. The generated raster filename is a SHA256 prefix of `scene_id`. Do not use a map from a different acquisition.

In [2]:
from pathlib import Path
import hashlib
from src.analysis13.core import load_scenes
scenes=load_scenes(ROOT)
REFERENCE_SCENE_ID = None  # Paste the exact scene_id from scenes CSV
REFERENCE_RASTER = None    # Path to its data/processed/analysis13_scene_rasters/...matched...tif
print('Available matched raster files:')
for item in sorted((ROOT/'data/processed/analysis13_scene_rasters').glob('*_matched_30m_EPSG32646.tif')):
    print(item.name)
print('Reference image should be independently interpreted near the same acquisition date.')

Available matched raster files:
877454787fe4_matched_30m_EPSG32646.tif
fc350713d26d_matched_30m_EPSG32646.tif
Reference image should be independently interpreted near the same acquisition date.


## Find nearby independently interpretable Sentinel-2 images

Optional Earth Engine search within ±10 days. The catalog cloud percentage is tile-wide, not haor-specific. Choose imagery with independently inspected clear pixels; SCL=water is NOT reference truth.

In [3]:
from src.analysis13.reference_ee import candidate_s2, export_s2_review_candidate
SEARCH_REFERENCE_CANDIDATES=False
if SEARCH_REFERENCE_CANDIDATES and REFERENCE_SCENE_ID:
    target=scenes.loc[scenes.scene_id==REFERENCE_SCENE_ID,'acquisition_local']
    assert len(target)==1, 'Exact source scene ID required'
    candidates=candidate_s2(ROOT,str(target.iloc[0]),window_days=10)
    display(candidates)
    # After choosing a genuinely suitable close-date image, call:
    # output=export_s2_review_candidate(ROOT,candidates.iloc[0].image_id)
else:
    print('Reference-image search deferred until a specific SAR scene is selected.')

Reference-image search deferred until a specific SAR scene is selected.


## Create blinded reference-labelling sample

Only samples a real matched scene raster. Private design table retains map-derived stratum; blind table excludes its predicted class. No class labels are generated.

In [4]:
from src.analysis13.validation import make_sample
if REFERENCE_SCENE_ID and REFERENCE_RASTER:
    token=hashlib.sha256(REFERENCE_SCENE_ID.encode()).hexdigest()[:12]
    assert Path(REFERENCE_RASTER).name.startswith(token), 'Raster filename does not match selected scene ID'
    design=make_sample(ROOT,Path(REFERENCE_RASTER),scene_id=REFERENCE_SCENE_ID,n_each=60)
    print(design)
else:
    print('Blocked: choose actual source scene/raster after 13B. No fabricated validation labels.')

Blocked: choose actual source scene/raster after 13B. No fabricated validation labels.


## Only score after independent annotation

Accepted reference labels: open_water, flooded_vegetation, dry_or_other, unobservable. Unobservable or missing cases block full-domain accuracy inference, instead of being silently excluded.

In [7]:
from src.analysis13.validation import score_reference
RUN_REFERENCE_SCORING=True
if RUN_REFERENCE_SCORING:
    print(score_reference(ROOT))
else:
    print('Awaiting independently interpreted, sourced reference-image labels')

FileNotFoundError: Need the generated sample metadata and independently labelled CSV

results\figures\biomass_20261002_reference_gamma0_four_polarizations.png
results\github_release\figures\boundary_QA_00_reference_extent_NOT_WETLAND_BOUNDARY.png
results\maps\00_reference_extent_NOT_WETLAND_BOUNDARY.png
results\tables\analysis13\13A_fixed_reference_points.csv


In [8]:
import ee
import pandas as pd

ee.Initialize(project="axial-paratext-491911-b6")

dates = [
    "2022-05-11", "2022-05-23",
    "2023-06-11", "2023-06-23",
    "2024-06-05", "2024-06-17", "2024-06-29",
    "2025-05-22", "2025-06-03",
]

# Uses the saved fixed-reference points to define a candidate search footprint.
# This is only for optical scene discovery, not the final validation polygon.
points = pd.read_csv(
    ROOT / "results/tables/analysis13/13A_fixed_reference_points.csv"
)

coords = points[["longitude", "latitude"]].values.tolist()
roi = ee.Geometry.MultiPoint(coords).bounds().buffer(1000)

collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

records = []

for date in dates:
    start = pd.Timestamp(date) - pd.Timedelta(days=5)
    end = pd.Timestamp(date) + pd.Timedelta(days=6)

    images = (
        collection
        .filterBounds(roi)
        .filterDate(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
        .sort("CLOUDY_PIXEL_PERCENTAGE")
        .limit(5)
    )

    for image in images.getInfo().get("features", []):
        props = image["properties"]
        records.append({
            "sar_date": date,
            "sentinel2_id": image["id"],
            "optical_date": pd.to_datetime(
                props["system:time_start"], unit="ms"
            ).strftime("%Y-%m-%d"),
            "scene_cloud_percent": props.get("CLOUDY_PIXEL_PERCENTAGE"),
        })

df = pd.DataFrame(records)

output = ROOT / "results/tables/analysis13/13E_sentinel2_candidates.csv"
df.to_csv(output, index=False)

display(df)
print("Saved:", output)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


,sar_date,sentinel2_id,optical_date,scene_cloud_percent
0,2022-05-11,COPERNICUS/S2_SR_HARMONIZED/20220508T041549_20...,2022-05-08,6.351554
1,2022-05-11,COPERNICUS/S2_SR_HARMONIZED/20220508T041549_20...,2022-05-08,16.333102
2,2022-05-11,COPERNICUS/S2_SR_HARMONIZED/20220506T042711_20...,2022-05-06,66.687584
3,2022-05-11,COPERNICUS/S2_SR_HARMONIZED/20220511T042659_20...,2022-05-11,87.032211
4,2022-05-11,COPERNICUS/S2_SR_HARMONIZED/20220506T042711_20...,2022-05-06,91.895300
5,2022-05-23,COPERNICUS/S2_SR_HARMONIZED/20220528T041549_20...,2022-05-28,32.876036
6,2022-05-23,COPERNICUS/S2_SR_HARMONIZED/20220528T041549_20...,2022-05-28,57.076722
7,2022-05-23,COPERNICUS/S2_SR_HARMONIZED/20220526T042711_20...,2022-05-26,73.318839
8,2022-05-23,COPERNICUS/S2_SR_HARMONIZED/20220526T042711_20...,2022-05-26,80.153668
9,2022-05-23,COPERNICUS/S2_SR_HARMONIZED/20220518T041549_20...,2022-05-18,94.569939


Saved: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\results\tables\analysis13\13E_sentinel2_candidates.csv


In [9]:
import ee
import pandas as pd

ee.Initialize(project="axial-paratext-491911-b6")

tables = ROOT / "results/tables/analysis13"

candidates = pd.read_csv(
    tables / "13E_sentinel2_candidates.csv"
)

points = pd.read_csv(
    tables / "13A_fixed_reference_points.csv"
)

features = [
    ee.Feature(
        ee.Geometry.Point([float(r.longitude), float(r.latitude)]),
        {"point_id": int(i)}
    )
    for i, r in points.reset_index(drop=True).iterrows()
]

reference_points = ee.FeatureCollection(features)

# Select three candidate images per SAR date.
selected = (
    candidates
    .sort_values(["sar_date", "scene_cloud_percent"])
    .groupby("sar_date", group_keys=False)
    .head(3)
)

records = []

# Sentinel-2 SCL categories:
# 4 = vegetation, 5 = bare surfaces, 6 = water.
# Cloud, haze, shadow, unclassified and snow are excluded.
usable_scl = {4, 5, 6}

for row in selected.itertuples(index=False):
    print("Checking:", row.sar_date, row.optical_date,
          row.sentinel2_id.split("/")[-1])

    image = ee.Image(row.sentinel2_id).select("SCL")

    samples = image.sampleRegions(
        collection=reference_points,
        scale=20,
        geometries=False
    ).getInfo()["features"]

    values = [
        feature["properties"].get("SCL")
        for feature in samples
    ]

    valid_values = [v for v in values if v is not None]

    clear_count = sum(
        int(v) in usable_scl
        for v in valid_values
    )

    records.append({
        "sar_date": row.sar_date,
        "optical_date": row.optical_date,
        "sentinel2_id": row.sentinel2_id,
        "scene_cloud_percent": row.scene_cloud_percent,
        "offset_days": (
            pd.Timestamp(row.optical_date)
            - pd.Timestamp(row.sar_date)
        ).days,
        "total_reference_points": len(points),
        "covered_points": len(valid_values),
        "usable_points": clear_count,
        "usable_fraction_all_points": (
            clear_count / len(points)
        ),
    })

qa = pd.DataFrame(records)

qa = qa.sort_values(
    ["sar_date", "usable_fraction_all_points"],
    ascending=[True, False]
)

out = tables / "13E_sentinel2_point_coverage_QA.csv"
qa.to_csv(out, index=False)

display(
    qa[[
        "sar_date",
        "optical_date",
        "offset_days",
        "scene_cloud_percent",
        "covered_points",
        "usable_points",
        "usable_fraction_all_points",
    ]]
)

print("Saved:", out)

Checking: 2022-05-11 2022-05-08 20220508T041549_20220508T042841_T46RDN
Checking: 2022-05-11 2022-05-08 20220508T041549_20220508T042841_T46RCN
Checking: 2022-05-11 2022-05-06 20220506T042711_20220506T043638_T46RCN
Checking: 2022-05-23 2022-05-28 20220528T041549_20220528T042711_T46RDN
Checking: 2022-05-23 2022-05-28 20220528T041549_20220528T042711_T46RCN
Checking: 2022-05-23 2022-05-26 20220526T042711_20220526T043829_T46RCN
Checking: 2023-06-11 2023-06-10 20230610T042711_20230610T043546_T46RDN
Checking: 2023-06-11 2023-06-10 20230610T042711_20230610T043546_T46RCN
Checking: 2023-06-11 2023-06-07 20230607T041551_20230607T042633_T46RDN
Checking: 2023-06-23 2023-06-25 20230625T042709_20230625T043753_T46RCN
Checking: 2023-06-23 2023-06-27 20230627T041551_20230627T042633_T46RCN
Checking: 2023-06-23 2023-06-27 20230627T041551_20230627T042633_T46RDN
Checking: 2024-06-05 2024-06-01 20240601T041551_20240601T043107_T46RDN
Checking: 2024-06-05 2024-06-01 20240601T041551_20240601T043107_T46RCN
Checki

,sar_date,optical_date,offset_days,scene_cloud_percent,covered_points,usable_points,usable_fraction_all_points
0,2022-05-11,2022-05-08,-3,6.351554,219,200,0.363636
1,2022-05-11,2022-05-08,-3,16.333102,201,182,0.330909
2,2022-05-11,2022-05-06,-5,66.687584,532,36,0.065455
4,2022-05-23,2022-05-28,5,57.076722,237,198,0.360000
3,2022-05-23,2022-05-28,5,32.876036,255,164,0.298182
5,2022-05-23,2022-05-26,3,73.318839,532,11,0.020000
6,2023-06-11,2023-06-10,-1,26.547676,473,182,0.330909
7,2023-06-11,2023-06-10,-1,55.036086,532,146,0.265455
8,2023-06-11,2023-06-07,-4,58.154923,296,0,0.000000
9,2023-06-23,2023-06-25,2,61.761254,532,86,0.156364


Saved: F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project\results\tables\analysis13\13E_sentinel2_point_coverage_QA.csv
